# Constrained dispatch forecasting — hist (2021–2026), Colab GPU

Runtime > Change runtime type > **GPU** (A100/L4 if offered). The repo ships the
prepared training data (`data/preprocessed/hist/...`, 17 MB) and the mined stress
episodes, so cloning is all the setup there is. Every training run checkpoints per
epoch and auto-resumes — a disconnect costs at most one epoch; just re-run the cell.

## The problem

Predict the 5-min VIC dispatch vector **P** = (hydro, coal, gas_steam, gas_ocgt,
batt_chg, batt_dis) one step ahead, such that every prediction **satisfies the grid
physics by construction**:

- **balance** — predicted net demand equals the signed dispatch sum:
  `SIGN·P − D_t = 0` with `SIGN = (1,1,1,1,−1,1)` (battery charging is a load)
- **ramps** — per-generator asymmetric limits `−R_dn ≤ P_i,t − P_i,t−1 ≤ R_up`
  (data-derived, `ml/check_caps.py::RAMPS`)
- **floors** — `P_i ≥ 0` (without this, idle gas_steam with ramp-down 500 could
  legally emit −500 MW)

Output vector everywhere below: **y = (D_t, P_1..P_6)** — 7 numbers, demand first.
Two hard-constraint mechanisms are trained and compared against their bare
backbones and against 5-min persistence (hist test bar: **WAPE 0.1084**).

## Approach 1 — RAYEN hard layer (`--arch *_rayen`), trained through

Konstantinov/RAYEN-style feasible-set parameterisation (`rayen-hardconvex.pdf`,
`lib/models.py::RayenHead`) replacing the network's final layer:

1. **anchor** `p = (SIGN·P_t−1, P_t−1)` — the previous step's actual dispatch. It
   satisfies everything at once: its sum equals its demand entry, zero change
   respects every ramp, and it is ε-interior to the floors (ε = 0.5 MW).
2. backbone emits a **raw direction** r ∈ R⁷ and a scalar s (8 raw outputs; the
   iTransformer keeps input RevIN but output denorm OFF — window means added onto
   direction components would corrupt them).
3. **stay on the balance plane**: r ← r − (a·r / a·a)·a with a = (−1, SIGN).
4. **respect walls**: α\* = distance along r to the nearest ramp/floor wall.
5. **output** y = p + σ(s)·α\*·r — feasible at any weights, exact gradients.

Zero direction = persistence, so the net learns bounded constrained *deviations*
from the strongest baseline. Trained with MSE on the 7-dim target
[nd_t (x-scaled), 6 dispatch (y-scaled)].

## Approach 2 — decision rules (`--arch *_task7` + post-hoc blend)

Constante Flores / Chen / Li 2025 (`hardlinearconstraints.pdf`,
`lib/decision_rule.py`). Two subnetworks, blended per prediction:

- **task network** — the same backbones with a plain 7-output head, trained for
  accuracy as usual (no constraint machinery in training). At inference its raw
  ŷ is snapped onto the balance plane by the closed-form orthogonal projection
  (paper eq. 16): ŷ ← ŷ − a(aᵀŷ)/(aᵀa).
- **safe network** — a fixed linear rule y_SN = F·x with x = [1, P_t−1, nd_t−1],
  found ONCE by the LP (paper eq. 14, per-row LP duality): maximise the
  worst-case slack t over the whole input box X subject to `aᵀF = 0` and every
  ramp/floor inequality holding for **all** x ∈ X. Solved by scipy/HiGHS in
  seconds; certificate t\* = **29.45 MW** (binding wall: gas_steam R_up/2), and
  the solution is interpretable: persistence + fixed positive offsets.
- **blend** — per sample, α = max over violated inequalities of
  −s_TN/(s_SN − s_TN) (paper eq. 4; α = 0 when the task output violates
  nothing). y = (1−α)·y_TN + α·y_SN is feasible by convexity; the equality is
  preserved because both endpoints satisfy it.

The training script fits the LP and reports the +DR row automatically after each
`*_task7` run; the bare task-net row doubles as the unconstrained Pareto baseline.

## Evaluation protocol

| check | where | why |
| --- | --- | --- |
| demand balance | regular test set, teacher-forced (Jan–Jul 2026, 53,568 windows) | balance is per-step; TF isolates the mechanism |
| ramp rate | **simulated test set**: closed-loop autoregressive rollout over the 15 test-split stress episodes (`constraints/stress_episodes.json` — record peaks, price spikes, battery-cycling days) | in deployment ramps bind on the model's *own* trajectory |

Metrics (`constraints/eval_hist_models.py`): WAPE (+ per-target), nd_WAPE (D_t
forecast quality), `n_demand` @own / @act (steps with |mismatch| > 10 MW),
`mismatch_act_pct` = 100·Σ|SIGN·pred − nd_act| / Σ|nd_act|, `n_ramp` (+0.6 MW
tol), `ramp_excess_pct` = 100·Σ(MW beyond limit) / Σ|pred| (raw violation MWh as
a share of dispatched MWh), `n_neg`, and battery feasibility = per-day (TF) /
per-episode (CL) SOC-window existence at η_rt = 0.834, cap 4,736 MWh.

In [20]:
REPO = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""  # repo is PRIVATE: paste a fine-grained READ-ONLY token (Contents: read)
import os
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.exists("energy_modelling"):
    !git clone -q $url
%cd energy_modelling
!git pull -q
!nvidia-smi -L  # scipy/HiGHS for the safe-network LP ships with Colab

/content/energy_modelling/energy_modelling/energy_modelling
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-8b6daf7f-6a36-1b4c-7a41-572060e35593)


In [21]:
# optional: mount Drive so weights/metrics survive the VM and sync back to you
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/energy_runs"
os.makedirs(OUT, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0 · Sanity: constraint layers are feasible at random weights

The guarantees are architectural — they must hold before a single gradient step.
~2 min; also certifies the safe-network LP (expect `t* = 29.45 MW`).

In [22]:
!python constraints/test_constraint_layers.py

test_dr_adversarial:
  DR/adversarial0 (alpha max 1.000): bal_max=0.0014 MW, ramp/floor OK (min P=33.3716)
  DR/adversarial1 (alpha max 1.000): bal_max=0.0014 MW, ramp/floor OK (min P=-0.0113)
  DR/adversarial2 (alpha max 1.000): bal_max=0.0010 MW, ramp/floor OK (min P=41.4069)
test_dr_noop_when_feasible:
  DR no-op: alpha_max=4.7e-06, max drift 6.77e-05
test_dr_random_weights:
  DR/lstm (alpha act 72%): bal_max=0.0010 MW, ramp/floor OK (min P=6.6867)
/content/energy_modelling/energy_modelling/energy_modelling/lib/models.py:502: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
  DR/itransformer (alpha act 86%): bal_max=0.0011 MW, ramp/floor OK (min P=-0.0000)
test_lp_certificate:
  LP: t*=29.45 MW, mc_min_slack=29.45, mc_eq_max=4.1e-12
test_rayen_gradients:
  rayen gradients: 12/12 params nonzero
test_rayen_random_weights:
  rayen/lstm/s0: bal_max=0

## 1 · Strided pilots (~10 min total)

Protocol (plan.md): trained-through layers have failed here before, so every arch
gets a cheap stride-12 pilot before a full run. Look for: rayen self-check shows
`ramp_viol=0, neg=0`, balance residual ~1e-3 MW; task7 +DR row likewise; and no
val-loss explosion. Pilot WAPEs are optimistic-noisy — do not compare them.

In [ ]:
for arch in ["lstm_rayen", "itransformer_rayen", "lstm_task7", "itransformer_task7"]:
    !python ml/train_hist.py --arch $arch --seed 0 --stride 12 --epochs 8 --out /content/pilots

## 2 · Full training runs

Recipes (apples-to-apples with the reference models): lstm\*: 40 epochs, patience
6, batch 128; itransformer\*: 60 / 10 / 64; lr 1e-4. On an A100 expect roughly
20–40 min per LSTM and 1–2 h per iTransformer. Each cell is independent —
re-running resumes from the last epoch checkpoint. `*_task7` cells print three
result blocks: bare task net, safe-LP certificate, and the +DR constrained row.

In [23]:
# approach 1 — RAYEN hard layer, LSTM backbone
!python ml/train_hist.py --arch lstm_rayen --seed 0 --out $OUT

lstm_rayen_hist_s0: device=cuda recipe={'epochs': 40, 'patience': 10, 'batch': 128}
windows: train 393,984  val 52,992  test 53,568
  resumed at epoch 24 (best_val=0.40188)
  epoch 024  val_mse=0.40628  (31s)  early stop
/content/energy_modelling/energy_modelling/energy_modelling/lib/evaluate.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  xb = torch.from_numpy(X[i:i + batch]).to(device)
  rayen self-check: |SIGN.P - D| max=0.0011 MW, ramp_viol=0, neg=0, nd_WAPE=0.0181
lstm_rayen_hist_s0: WAPE=0.1454 R2=0.9668 (24 epochs, 1 min)
wrote /content/drive/MyDrive/energy_runs/lstm_rayen_hist_s0_metrics.json


In [24]:
# approach 1 — RAYEN hard layer, iTransformer (input RevIN, output denorm off)
!python ml/train_hist.py --arch itransformer_rayen --seed 0 --out $OUT

itransformer_rayen_hist_s0: device=cuda recipe={'epochs': 40, 'patience': 10, 'batch': 128}
windows: train 393,984  val 52,992  test 53,568
/content/energy_modelling/energy_modelling/energy_modelling/lib/models.py:502: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
  resumed at epoch 40 (best_val=0.39522)
  epoch 040  val_mse=0.39670  (40s)
/content/energy_modelling/energy_modelling/energy_modelling/lib/evaluate.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  xb = torch.f

In [25]:
# approach 2 — task net LSTM (bare row = Pareto baseline; +DR row = decision rules)
!python ml/train_hist.py --arch lstm_task7 --seed 0 --out $OUT

lstm_task7_hist_s0: device=cuda recipe={'epochs': 40, 'patience': 10, 'batch': 128}
windows: train 393,984  val 52,992  test 53,568
  resumed at epoch 35 (best_val=0.57802)
  epoch 035  val_mse=0.60307  (29s)  early stop
/content/energy_modelling/energy_modelling/energy_modelling/lib/evaluate.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  xb = torch.from_numpy(X[i:i + batch]).to(device)
  task7 bare: |SIGN.P - D| max=155.6651 MW, ramp_viol=731, neg=55158, nd_WAPE=0.0248
  safe LP: t*=29.45 MW  mc_min_slack=29.45 mc_eq_max=4.09e-12
  task7 +DR: |SIGN.P - D| max=0.0016 MW, ramp_viol=0, neg=0, nd_WAPE=0.0287
  t

In [26]:
# approach 2 — task net iTransformer (full RevIN: outputs are levels, not directions)
!python ml/train_hist.py --arch itransformer_task7 --seed 0 --out $OUT

itransformer_task7_hist_s0: device=cuda recipe={'epochs': 40, 'patience': 10, 'batch': 128}
windows: train 393,984  val 52,992  test 53,568
/content/energy_modelling/energy_modelling/energy_modelling/lib/models.py:502: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
  resumed at epoch 20 (best_val=0.37184)
  epoch 020  val_mse=0.38104  (33s)  early stop
/content/energy_modelling/energy_modelling/energy_modelling/lib/evaluate.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  

## 3 · Evaluation — both tables

One command: persistence benchmark + every checkpoint found in `$OUT`
(rayen models, bare task nets, +DR rows rebuilt from the saved safe-F).

- **TF table** (demand balance, regular test set) → `constraints/results/hist_tf.md`
- **CL table** (ramp rate, simulated test set = closed-loop stress episodes)
  → `constraints/results/hist_cl_stress.md`

Reading guide: constrained rows must show `bal_own_max ≈ 0`, `n_ramp = 0`,
`n_neg = 0` — the interesting question is their WAPE vs the bare rows and the
0.1084 persistence bar. `alpha_frac_active` (in the JSONs) tells how often the DR
safe net actually intervened on a converged task net (expect ≈ 0).

In [27]:
!python constraints/eval_hist_models.py --ckpt-dir $OUT
from IPython.display import Markdown, display
for t in ["hist_tf.md", "hist_cl_stress.md"]:
    display(Markdown(open(f"constraints/results/{t}").read()))

/content/energy_modelling/energy_modelling/energy_modelling/lib/models.py:502: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
  lstm_task7+DR: safe LP t*=29.45 MW
  itransformer_task7+DR: safe LP t*=29.45 MW
models: ['persistence', 'lstm_rayen', 'itransformer_rayen', 'lstm_task7', 'lstm_task7+DR', 'itransformer_task7', 'itransformer_task7+DR']  device=cuda
  TF persistence            WAPE=0.1084 nd_WAPE=0.0182 bal_own=0.000 dem_act=47728 (1.82%) neg=0 ramp=0 soc_day=100%
  TF lstm_rayen             WAPE=0.1454 nd_WAPE=0.0181 bal_own=0.001 dem_act=47725 (1.81%) neg=0 ramp=0 soc_day=100%
  TF itransformer_rayen     WAPE=0.1292 nd_WAPE=0.0182 bal_own=0.002 dem_act=47768 (1.82%) neg=0 ramp=0 soc_day=100%
  TF lstm_task7             WAPE=0.2699 nd_WAPE=0.0248 bal_own=155.665 dem_act=48297 (2.51%) neg=55158 ramp=731 soc_day=100%
  TF lstm_task7+DR      

# Hist teacher-forced test (demand balance)

Test = Jan-Jul 2026 (stride 1). @own = vs the model's own D_t (mechanism check, threshold 10.0 MW); @act = vs actual nd(t). nd_WAPE = D_t forecast quality. SOC per calendar day, eta_rt=0.834, cap 4736 MWh.

| model | WAPE | R2 | nd_WAPE | bal_own_max_mw | n_demand_own | n_demand_act | mismatch_act_pct | n_neg | n_ramp_vs_prev | soc_day_feasible_pct | soc_worst_day_pct |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| persistence | 0.1084 | 0.9669 | 0.0182 | 0.0004 | 0 | 47728 | 1.8158 | 0 | 0 | 100.0 | 79.4 |
| lstm_rayen | 0.1454 | 0.9668 | 0.0181 | 0.0011 | 0 | 47725 | 1.8102 | 0 | 0 | 100.0 | 79.7 |
| itransformer_rayen | 0.1292 | 0.9674 | 0.0182 | 0.0017 | 0 | 47768 | 1.8176 | 0 | 0 | 100.0 | 76.9 |
| lstm_task7 | 0.2699 | 0.9431 | 0.0248 | 155.7 | 16593 | 48297 | 2.5053 | 55158 | 731 | 100.0 | 61.9 |
| lstm_task7+DR | 0.3201 | 0.8992 | 0.0287 | 0.0016 | 0 | 49217 | 2.8713 | 0 | 0 | 100.0 | 71.9 |
| itransformer_task7 | 0.1313 | 0.9683 | 0.0211 | 254.3 | 38495 | 48766 | 1.9774 | 35505 | 0 | 100.0 | 76.2 |
| itransformer_task7+DR | 0.2692 | 0.9299 | 0.0271 | 0.0016 | 0 | 49925 | 2.7052 | 0 | 0 | 100.0 | 77.2 |


# Hist closed-loop stress episodes (ramp rate)

Autoregressive rollout over TEST-split stress episodes; models feed back their own dispatch, exogenous features stay actual. Ramps on the model's own trajectory (asymmetric data limits + 0.6 MW tol); ramp_excess_pct = 100*sum(MW beyond limit)/sum(|pred|). SOC per episode.

| model | cl_WAPE | n_ramp | ramp_excess_pct | bal_own_max_mw | n_demand_act | mismatch_act_pct | n_neg | soc_feasible_eps | soc_worst_ep_pct |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| persistence | 0.7589 | 0 | 0.0000 | 0.0000 | 4286 | 23.5 | 0 | 12 | 239.4 |
| lstm_rayen | 0.7988 | 0 | 0.0000 | 0.0009 | 4273 | 23.3 | 0 | 14 | 111.1 |
| itransformer_rayen | 0.7642 | 0 | 0.0000 | 0.0010 | 4282 | 23.5 | 0 | 13 | 240.6 |
| lstm_task7 | 0.6334 | 0 | 0.0000 | 78.9 | 4069 | 3.5178 | 5646 | 15 | 23.7 |
| lstm_task7+DR | 3.2141 | 0 | 0.0000 | 0.0014 | 4234 | 31.6 | 0 | 9 | 125.7 |
| itransformer_task7 | 1.3554 | 84 | 0.0453 | 48905.6 | 4301 | 97.2 | 5226 | 15 | 85.7 |
| itransformer_task7+DR | 294.1 | 0 | 0.0000 | 0.1124 | 4320 | 3141.5 | 0 | 0 | 628.6 |


In [ ]:
# no Drive? zip and download instead
# !zip -r runs.zip $OUT constraints/results && python -c "from google.colab import files; files.download('runs.zip')"